# Stage 8: Market Recommendation Engine
## Smart Mandi Advisory for Farmers — Nagpur District (2022–2026)

### Purpose & Problem Statement:
Smallholder farmers in Vidarbha / Nagpur often sell their harvest to the nearest local middleman or nearest mandi without knowing that a neighboring mandi just 20–40 km away might be offering ₹500–₹1,000 more per quintal. 

This **Market Recommendation Engine** operationalizes our machine learning model:
1. **Forecasts 1-day ahead prices** across all 13 APMC mandis simultaneously.
2. **Deducts transportation & logistics costs** dynamically based on diesel fuel rates and distance.
3. **Computes Net Realized Revenue** per quintal and total net payout.
4. **Verifies MSP Protection**: Flags if prices fall below the government Minimum Support Price.
5. **Ranks Mandis & Generates Actionable Advisory**: Recommends the optimal market with quantified monetary gain!


### Step 1: Import Recommender Module and Load Test Data


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

from recommendation_engine import MandiRecommender

model_path = "../7.model_training_and_evaluation/crop_price_prediction_model.joblib"
recommender = MandiRecommender(model_path)
print(" Recommender engine initialized with trained ML bundle.")

test_df = pd.read_csv("../6.train_test_split/test_data.csv")
test_df['date'] = pd.to_datetime(test_df['date'])
print(f"Loaded {len(test_df):,} test records across {test_df['commodity'].nunique()} commodities and {test_df['market'].nunique()} markets.")


### Step 2: Case Study 1 — Onion Farmer Selling 20 Quintals (Peak Summer)


In [ ]:
date_query = "2026-05-15"
crop_query = "Onion"
qty_query = 20  # 20 quintals (2 metric tonnes)

onion_slice = test_df[(test_df['date'] == date_query) & (test_df['commodity'] == crop_query)]
ranking_df, summary = recommender.recommend(onion_slice, quantity_quintals=qty_query)

print("=" * 85)
print(f"  FARMER ADVISORY: {crop_query.upper()} | Date: {date_query} | Volume: {qty_query} Quintals")
print("=" * 85)
cols = ['market', 'actual_price', 'predicted_modal_price', 'accuracy_pct', 'distance_km', 'transport_cost_per_q', 'net_realized_price', 'total_net_payout', 'gain_pct_vs_avg']
display = ranking_df[cols].copy()
display['accuracy_pct'] = display['accuracy_pct'].apply(lambda x: f"{x:.2f}%")
display.columns = ['Mandi', 'Actual (Rs/q)', 'Pred Price (Rs/q)', 'Accuracy', 'Dist (km)', 'Transport (Rs/q)', 'Net Price (Rs/q)', f'Total Net (Rs on {qty_query}q)', 'Advantage (%)']
print(display.to_string(index=True))

print("\n" + "-" * 85)
print(f" 🏆 RECOMMENDED MARKET: {summary['best_mandi'].upper()} APMC MANDI")
print(f" 💰 Total Net Payout Expected : Rs. {summary['total_best_payout']:,.2f}")
print(f" 📈 Extra Profit Earned vs Lowest Mandi ({summary['worst_mandi']}) : +Rs. {summary['profit_advantage']:,.2f} !")
print(f" 🛡️ MSP Floor Status : {'Above MSP (Safe)' if summary['is_above_msp'] else 'Below MSP (Hold harvest)'}")
print("-" * 85)


### Step 3: Case Study 2 — Cotton Farmer Selling 50 Quintals (Kharif Cash Crop)


In [ ]:
date_query = "2026-02-10"
crop_query = "Cotton"
qty_query = 50  # 50 quintals

cotton_slice = test_df[(test_df['date'] == date_query) & (test_df['commodity'] == crop_query)]
ranking_df, summary = recommender.recommend(cotton_slice, quantity_quintals=qty_query)

print("=" * 85)
print(f"  FARMER ADVISORY: {crop_query.upper()} | Date: {date_query} | Volume: {qty_query} Quintals")
print("=" * 85)
display = ranking_df[cols].copy()
display.columns = ['Mandi', 'Pred Price (Rs/q)', 'Dist (km)', 'Transport (Rs/q)', 'Net Price (Rs/q)', f'Total Net (Rs on {qty_query}q)', 'Advantage (%)']
print(display.to_string(index=True))

print("\n" + "-" * 85)
print(f" 🏆 RECOMMENDED MARKET: {summary['best_mandi'].upper()} APMC MANDI")
print(f" 💰 Total Net Payout Expected : Rs. {summary['total_best_payout']:,.2f}")
print(f" 📈 Extra Profit Earned vs Lowest Mandi ({summary['worst_mandi']}) : +Rs. {summary['profit_advantage']:,.2f} !")
print("-" * 85)


### Step 4: Case Study 3 — Tomato Farmer Selling 30 Quintals (Perishable Horticulture)


In [ ]:
date_query = "2026-08-14"
crop_query = "Tomato"
qty_query = 30  # 30 quintals

tomato_slice = test_df[(test_df['date'] == date_query) & (test_df['commodity'] == crop_query)]
ranking_df, summary = recommender.recommend(tomato_slice, quantity_quintals=qty_query)

print("=" * 85)
print(f"  FARMER ADVISORY: {crop_query.upper()} | Date: {date_query} | Volume: {qty_query} Quintals")
print("=" * 85)
display = ranking_df[cols].copy()
display.columns = ['Mandi', 'Pred Price (Rs/q)', 'Dist (km)', 'Transport (Rs/q)', 'Net Price (Rs/q)', f'Total Net (Rs on {qty_query}q)', 'Advantage (%)']
print(display.to_string(index=True))

print("\n" + "-" * 85)
print(f" 🏆 RECOMMENDED MARKET: {summary['best_mandi'].upper()} APMC MANDI")
print(f" 💰 Total Net Payout Expected : Rs. {summary['total_best_payout']:,.2f}")
print(f" 📈 Extra Profit Earned vs Lowest Mandi ({summary['worst_mandi']}) : +Rs. {summary['profit_advantage']:,.2f} !")
print("-" * 85)


### Step 5: District-Wide Profitability Simulation Across All Crops
Calculate average profit gain across the entire test dataset when farmers use our Market Recommendation Engine instead of selling at the worst local market.


In [ ]:
print("Running simulation across multiple sample dates...")
sample_dates = test_df['date'].drop_duplicates().sample(15, random_state=42)

sim_results = []
for d in sample_dates:
    for crop in test_df['commodity'].unique():
        s = test_df[(test_df['date'] == d) & (test_df['commodity'] == crop)]
        if len(s) == 13:
            _, sm = recommender.recommend(s, quantity_quintals=25)
            sim_results.append({
                "Date": str(d.date()),
                "Commodity": crop,
                "Best Mandi": sm['best_mandi'],
                "Worst Mandi": sm['worst_mandi'],
                "Net Realized (Best)": sm['net_best_price'],
                "Net Realized (Worst)": round(sm['total_worst_payout'] / 25, 2),
                "Extra Profit (25 Quintals)": sm['profit_advantage']
            })

sim_df = pd.DataFrame(sim_results)
avg_extra_gain = sim_df['Extra Profit (25 Quintals)'].mean()
print(f"Total simulated scenarios : {len(sim_df)}")
print(f"Average Extra Profit earned per farmer on 25 quintals: Rs. {avg_extra_gain:,.2f} !")
print(f"Maximum Single Sale Gain observed                   : Rs. {sim_df['Extra Profit (25 Quintals)'].max():,.2f} !")

sim_df.to_csv("simulation_recommendation_results.csv", index=False)
print(" Simulation results successfully exported to 'simulation_recommendation_results.csv'.")
